In [ ]:

# ============================================================
# LS-SSDD Fine-tuned YOLO26-S
# → HRSID Official Test Set Evaluation
#
# Standalone version
# - Download/copy HRSID from Google Drive
# - Convert official HRSID test2017.json to YOLO format
# - Create data.yaml
# - Load LS-SSDD fine-tuned model
# - Evaluate on HRSID official test set
# ============================================================


# ============================================================
# 1. Install & Import
# ============================================================

!pip install -q -U ultralytics

import json
import shutil
from pathlib import Path
from collections import defaultdict

from ultralytics import YOLO
from google.colab import drive


# ============================================================
# 2. Mount Google Drive
# ============================================================

drive.mount("/content/drive")


# ============================================================
# 3. Path Setting
# ============================================================

# Original HRSID dataset in Google Drive
DRIVE_HRSID_DIR = Path(
    "/content/drive/MyDrive/"
    "SAR_AI_Ship_Detection/"
    "HRSID"
)

# Copy HRSID to Colab local storage
LOCAL_HRSID_DIR = Path("/content/HRSID")

# YOLO-format HRSID test dataset
YOLO_DIR = Path("/content/HRSID-YOLO-EVAL")

# Original HRSID folders
IMG_DIR = LOCAL_HRSID_DIR / "images"
ANN_DIR = LOCAL_HRSID_DIR / "annotations"

# Official HRSID test annotation
TEST_JSON = ANN_DIR / "test2017.json"


# ============================================================
# 4. LS-SSDD Fine-tuned Model
# ============================================================

MODEL_PATH = Path(
      "/content/drive/MyDrive/"    "SAR_AI_Ship_Detection/"
    "trained_models/"
    "hrsid_yolo26s_20epoch_best.pt"
)


# ============================================================
# 5. Copy HRSID Drive → Colab
# ============================================================

assert DRIVE_HRSID_DIR.exists(), (
    f"HRSID Drive folder not found:\n{DRIVE_HRSID_DIR}"
)

if not LOCAL_HRSID_DIR.exists():

    print("Copying HRSID from Google Drive to /content ...")

    shutil.copytree(
        DRIVE_HRSID_DIR,
        LOCAL_HRSID_DIR
    )

    print("HRSID copied successfully.")

else:

    print("HRSID already exists in /content.")


# ============================================================
# 6. Check HRSID
# ============================================================

assert IMG_DIR.exists(), (
    f"Image directory not found:\n{IMG_DIR}"
)

assert TEST_JSON.exists(), (
    f"test2017.json not found:\n{TEST_JSON}"
)

print("\nHRSID image directory:")
print(IMG_DIR)

print("\nHRSID official test JSON:")
print(TEST_JSON)


# ============================================================
# 7. Index HRSID Images
# ============================================================

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff"
}

image_index = {}

for p in IMG_DIR.rglob("*"):

    if (
        p.is_file()
        and p.suffix.lower() in IMAGE_EXTENSIONS
    ):

        image_index[p.name] = p


print("\nIndexed HRSID images:", len(image_index))


def find_image(file_name):

    # Exact filename
    if file_name in image_index:
        return image_index[file_name]

    # Fallback: search same stem
    stem = Path(file_name).stem

    for ext in IMAGE_EXTENSIONS:

        candidate = f"{stem}{ext}"

        if candidate in image_index:
            return image_index[candidate]

    return None


# ============================================================
# 8. COCO bbox → YOLO bbox
# ============================================================

def coco_bbox_to_yolo(
    bbox,
    img_width,
    img_height
):

    # COCO:
    # x_min, y_min, width, height

    x, y, w, h = bbox

    x_center = x + w / 2
    y_center = y + h / 2

    # Normalize
    x_center /= img_width
    y_center /= img_height

    w /= img_width
    h /= img_height

    # Clamp
    x_center = min(max(x_center, 0.0), 1.0)
    y_center = min(max(y_center, 0.0), 1.0)

    w = min(max(w, 0.0), 1.0)
    h = min(max(h, 0.0), 1.0)

    return (
        x_center,
        y_center,
        w,
        h
    )


# ============================================================
# 9. Load Official HRSID Test Annotation
# ============================================================

with open(TEST_JSON, "r") as f:

    coco = json.load(f)


test_images = {

    img["id"]: img

    for img in coco["images"]
}


test_annotations = defaultdict(list)

for ann in coco["annotations"]:

    test_annotations[
        ann["image_id"]
    ].append(ann)


print("\n" + "=" * 60)
print("Official HRSID Test Dataset")
print("=" * 60)

print(
    "Images      :",
    len(test_images)
)

print(
    "Annotations :",
    len(coco["annotations"])
)


# ============================================================
# 10. Reset YOLO Evaluation Dataset
# ============================================================

if YOLO_DIR.exists():

    shutil.rmtree(YOLO_DIR)


TEST_IMG_OUT = (
    YOLO_DIR
    / "images"
    / "test"
)

TEST_LABEL_OUT = (
    YOLO_DIR
    / "labels"
    / "test"
)


TEST_IMG_OUT.mkdir(
    parents=True,
    exist_ok=True
)

TEST_LABEL_OUT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 11. Convert HRSID Official Test → YOLO
# ============================================================

missing_images = 0
total_annotations = 0


for image_id, img_info in test_images.items():

    file_name = img_info["file_name"]

    width = img_info["width"]
    height = img_info["height"]

    # Find original image
    img_path = find_image(file_name)

    if img_path is None:

        print(
            "Missing image:",
            file_name
        )

        missing_images += 1

        continue


    # --------------------------------------------------------
    # Copy image
    # --------------------------------------------------------

    dst_img = (
        TEST_IMG_OUT
        / img_path.name
    )

    shutil.copy(
        img_path,
        dst_img
    )


    # --------------------------------------------------------
    # Convert annotations
    # --------------------------------------------------------

    anns = test_annotations.get(
        image_id,
        []
    )

    yolo_lines = []


    for ann in anns:

        bbox = ann["bbox"]

        (
            x_center,
            y_center,
            w,
            h

        ) = coco_bbox_to_yolo(

            bbox,
            width,
            height
        )


        # HRSID has one class: ship
        class_id = 0


        yolo_lines.append(

            f"{class_id} "
            f"{x_center:.6f} "
            f"{y_center:.6f} "
            f"{w:.6f} "
            f"{h:.6f}"
        )

        total_annotations += 1


    # Empty txt is also created for background images
    label_path = (
        TEST_LABEL_OUT
        / f"{img_path.stem}.txt"
    )

    with open(
        label_path,
        "w"
    ) as f:

        f.write(
            "\n".join(yolo_lines)
        )


# ============================================================
# 12. Check Conversion
# ============================================================

converted_images = list(
    TEST_IMG_OUT.glob("*")
)

converted_labels = list(
    TEST_LABEL_OUT.glob("*.txt")
)


print("\n" + "=" * 60)
print("Converted HRSID Test Dataset")
print("=" * 60)

print(
    "Images      :",
    len(converted_images)
)

print(
    "Labels      :",
    len(converted_labels)
)

print(
    "Annotations :",
    total_annotations
)

print(
    "Missing     :",
    missing_images
)


# Expected:
# Images      : 1962
# Labels      : 1962
# Annotations : 5922
# Missing     : 0

assert len(converted_images) == 1962, (
    f"Unexpected image count: {len(converted_images)}"
)

assert len(converted_labels) == 1962, (
    f"Unexpected label count: {len(converted_labels)}"
)

assert total_annotations == 5922, (
    f"Unexpected annotation count: {total_annotations}"
)

assert missing_images == 0, (
    f"Missing images: {missing_images}"
)


print("\nHRSID official test conversion OK.")


# ============================================================
# 13. Make data.yaml
# ============================================================

DATA_YAML = (
    YOLO_DIR
    / "data.yaml"
)


with open(
    DATA_YAML,
    "w"
) as f:

    f.write(
f"""
path: {YOLO_DIR}

# Evaluation-only dataset
train: images/test
val: images/test
test: images/test

names:
  0: ship
"""
    )


print("\n" + "=" * 60)
print("data.yaml")
print("=" * 60)

print(
    DATA_YAML.read_text()
)


# ============================================================
# 14. Check LS-SSDD Fine-tuned Model
# ============================================================

assert MODEL_PATH.exists(), (
    "LS-SSDD fine-tuned model not found:\n"
    f"{MODEL_PATH}\n\n"
    "Check the model path before evaluation."
)


print("=" * 60)
print("Evaluation Model")
print("=" * 60)

print(MODEL_PATH)


# ============================================================
# 15. Load Model
# ============================================================

model = YOLO(
    str(MODEL_PATH)
)


# ============================================================
# 16. Evaluate LS-SSDD Fine-tuned Model
#     on HRSID Official Test
# ============================================================

results = model.val(

    data=str(DATA_YAML),

    split="test",

    imgsz=800,

    batch=16,

    device=0,

    workers=2
)


# ============================================================
# 17. Print Results
# ============================================================

precision = results.box.mp
recall = results.box.mr

map50 = results.box.map50
map50_95 = results.box.map


print("\n" + "=" * 60)

print(
    "LS-SSDD Fine-tuned YOLO26-S"
)

print(
    "→ HRSID Official Test Results"
)

print("=" * 60)


print(
    f"Precision      : "
    f"{precision:.4f}"
)

print(
    f"Recall         : "
    f"{recall:.4f}"
)

print(
    f"mAP@0.5        : "
    f"{map50:.4f}"
)

print(
    f"mAP@0.5:0.95   : "
    f"{map50_95:.4f}"
)

print("=" * 60)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
HRSID already exists in /content.

HRSID image directory:
/content/HRSID/images

HRSID official test JSON:
/content/HRSID/annotations/test2017.json

Indexed HRSID images: 5606

Official HRSID Test Dataset
Images      : 1962
Annotations : 5922

Converted HRSID Test Dataset
Images      : 1962
Labels      : 1962
Annotations : 5922
Missing     : 0

HRSID official test conversion OK.

data.yaml

path: /content/HRSID-YOLO-EVAL

# Evaluation-only dataset
train: images/test
val: images/test
test: images/test

names:
  0: ship

Evaluation Model
/content/drive/MyDrive/SAR_AI_Ship_Detection/trained_models/hrsid_yolo26s_20epoch_best.pt
Ultralytics 8.4.135 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26s summary (fused): 122 layers, 9,465,567 parameters, 0 gradients, 20.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1802.6±1486.5 MB/s, s